## Example 1: Basic Python code parsing

In [ ]:
import parso

# Example 1: Basic Python Code Parsing
try:
    # Parse simple Python code
    code = "x = 42 + 3\nprint(x)"
    module = parso.parse(code)
    
    print(f"Parsed code: {repr(code)}")
    print(f"Module type: {type(module).__name__}")
    print(f"Module start position: {module.start_pos}")
    print(f"Module end position: {module.end_pos}")
    
    # Get children nodes
    children = module.children
    print(f"Number of statements: {len(children)}")
    
    for i, child in enumerate(children):
        if hasattr(child, 'type'):
            print(f"Statement {i}: {child.type} at {child.start_pos}")
    
    # Get the code back
    reconstructed = module.get_code()
    print(f"Reconstructed code: {repr(reconstructed)}")
    print(f"Code match: {reconstructed == code}")
    
    print("Example 1: Basic Python parsing completed")
    
except Exception as e:
    print(f"Error in basic parsing: {e}")
    raise AssertionError(
        f"parso example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Function and class parsing with tree traversal

In [ ]:
import parso

# Example 2: Function and Class Parsing with AST Traversal
try:
    # Parse more complex Python code with function and class
    code = '''
def calculate(x, y=10):
    """Calculate something."""
    return x * y + 1

class MyClass:
    def __init__(self, value):
        self.value = value
    
    def method(self):
        return self.value * 2
'''
    
    module = parso.parse(code)
    print("Parsed complex code with function and class")
    
    # Traverse the AST to find specific nodes
    def find_nodes_by_type(node, node_type):
        """Recursively find all nodes of a specific type"""
        found = []
        if hasattr(node, 'type') and node.type == node_type:
            found.append(node)
        if hasattr(node, 'children'):
            for child in node.children:
                found.extend(find_nodes_by_type(child, node_type))
        return found
    
    # Find function definitions
    functions = find_nodes_by_type(module, 'funcdef')
    print(f"Found {len(functions)} function definitions:")
    for func in functions:
        name = func.name.value if hasattr(func, 'name') else "unknown"
        print(f"  Function: {name} at line {func.start_pos[0]}")
    
    # Find class definitions
    classes = find_nodes_by_type(module, 'classdef')
    print(f"Found {len(classes)} class definitions:")
    for cls in classes:
        name = cls.name.value if hasattr(cls, 'name') else "unknown"
        print(f"  Class: {name} at line {cls.start_pos[0]}")
    
    # Find string literals (docstrings)
    strings = find_nodes_by_type(module, 'string')
    print(f"Found {len(strings)} string literals")
    
    print("Example 2: Complex code parsing and AST traversal completed")
    
except Exception as e:
    print(f"Error in complex parsing: {e}")
    raise AssertionError(
        f"parso example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Error handling and incomplete code parsing

In [ ]:
import parso

# Example 3: Error Handling and Incomplete Code Parsing
try:
    # Test parsing with syntax errors
    print("Testing error handling:")
    
    # Valid code first
    valid_code = "x = 1 + 2"
    valid_module = parso.parse(valid_code)
    print(f"Valid code parsed successfully: {valid_module.type}")
    
    # Test with syntax error
    invalid_code = "x = 1 +"  # Incomplete expression
    try:
        invalid_module = parso.parse(invalid_code)
        print(f"Invalid code still parsed: {invalid_module.type}")
        
        # Check for error nodes
        def find_error_nodes(node):
            errors = []
            if hasattr(node, 'type') and 'error' in node.type.lower():
                errors.append(node)
            if hasattr(node, 'children'):
                for child in node.children:
                    errors.extend(find_error_nodes(child))
            return errors
        
        error_nodes = find_error_nodes(invalid_module)
        print(f"Found {len(error_nodes)} error nodes")
        
    except Exception as parse_error:
        print(f"Parse error: {parse_error}")
        raise
    
    # Test incomplete code (parso is forgiving)
    incomplete_code = "def func():\n    x = "
    incomplete_module = parso.parse(incomplete_code)
    print(f"Incomplete code parsed: {incomplete_module.type}")
    
    # Different Python versions parsing: parso takes the version as a string like "3.8", not a
    # tuple. A bare `x := 42` statement is invalid in every version; the parenthesized form is
    # 3.8+. error_recovery=False makes parso raise ParserSyntaxError instead of building error nodes.
    for version in ["3.6", "3.8", "3.10"]:
        try:
            versioned_module = parso.parse("(x := 42)", version=version, error_recovery=False)  # Walrus operator (3.8+)
            print(f"Walrus operator parsed for Python {version}")
            assert version != "3.6", "the 3.6 grammar accepted the walrus operator"
        except parso.ParserSyntaxError:
            print(f"Walrus operator not supported for Python {version}")
            assert version == "3.6", f"the {version} grammar rejected the walrus operator"
    
    print("Example 3: Error handling and version parsing completed")
    
except Exception as e:
    print(f"Error in error handling test: {e}")
    raise AssertionError(
        f"parso example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Node properties and code analysis

In [ ]:
import parso

# Example 4: Node Properties and Code Analysis
try:
    # Parse code with various constructs for detailed analysis
    code = '''
import os
from collections import defaultdict

def process_data(items, multiplier=2):
    result = defaultdict(list)
    for item in items:
        if item > 0:
            result['positive'].append(item * multiplier)
        else:
            result['negative'].append(abs(item))
    return dict(result)

# Comment here
data = [1, -2, 3, -4, 5]
processed = process_data(data)
'''
    
    module = parso.parse(code)
    print("Analyzing detailed code structure:")
    
    # Analyze imports
    def analyze_imports(node):
        imports = []
        if hasattr(node, 'type'):
            if node.type in ['import_name', 'import_from']:
                imports.append(node)
        if hasattr(node, 'children'):
            for child in node.children:
                imports.extend(analyze_imports(child))
        return imports
    
    import_nodes = analyze_imports(module)
    print(f"Found {len(import_nodes)} import statements:")
    for imp in import_nodes:
        print(f"  Import at line {imp.start_pos[0]}: {imp.get_code().strip()}")
    
    # Analyze names/identifiers
    def find_names(node):
        names = []
        if hasattr(node, 'type') and node.type == 'name':
            names.append(node)
        if hasattr(node, 'children'):
            for child in node.children:
                names.extend(find_names(child))
        return names
    
    name_nodes = find_names(module)
    unique_names = set(name.value for name in name_nodes)
    print(f"Found {len(name_nodes)} name references, {len(unique_names)} unique:")
    print(f"  Names: {sorted(list(unique_names))[:10]}...")  # Show first 10
    
    # Find operators
    def find_operators(node):
        operators = []
        if hasattr(node, 'type') and 'operator' in node.type:
            operators.append(node)
        if hasattr(node, 'children'):
            for child in node.children:
                operators.extend(find_operators(child))
        return operators
    
    op_nodes = find_operators(module)
    print(f"Found {len(op_nodes)} operators")
    
    print("Example 4: Node properties and code analysis completed")
    
except Exception as e:
    print(f"Error in code analysis: {e}")
    raise AssertionError(
        f"parso example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Grammar and language features

In [ ]:
import parso

# Example 5: Grammar and Language Features
try:
    # Test different Python language constructs
    constructs = [
        ("List comprehension", "[x*2 for x in range(5) if x % 2 == 0]"),
        ("Dict comprehension", "{k: v*2 for k, v in {'a': 1, 'b': 2}.items()}"),
        ("Lambda function", "lambda x, y=10: x + y"),
        ("Generator expression", "(x for x in range(10) if x > 5)"),
        ("F-string", "f'Hello {name}, you are {age} years old'"),
        ("Async function", "async def fetch(): return await get_data()"),
        ("Context manager", "with open('file.txt') as f: content = f.read()"),
        ("Exception handling", "try:\n    result = 1/0\nexcept ZeroDivisionError:\n    result = 0"),
    ]
    
    print("Testing various Python language constructs:")
    
    for name, code in constructs:
        try:
            parsed = parso.parse(code)
            # parso leaves have get_next_leaf() / get_next_sibling() (no plural); count every leaf
            leaf, node_count = parsed.get_first_leaf(), 0
            while leaf is not None:
                node_count += 1
                leaf = leaf.get_next_leaf()
            assert parsed.get_code() == code, "parso must round-trip the source exactly"
            assert not list(parso.load_grammar().iter_errors(parsed)), f"{name} reported syntax errors"
            print(f"✓ {name}: parsed successfully ({node_count} leaves)")
        except Exception as e:
            print(f"✗ {name}: parsing failed - {e}")
            raise
    
    # Test grammar versions
    print(f"\nParso version: {parso.__version__}")
    
    # Create grammar for specific Python version
    grammar_36 = parso.load_grammar(version='3.6')
    grammar_39 = parso.load_grammar(version='3.9')
    
    print(f"Grammar 3.6 type: {type(grammar_36).__name__}")
    print(f"Grammar 3.9 type: {type(grammar_39).__name__}")
    
    # Test version-specific features
    walrus_code = "if (n := len(items)) > 10: print(f'Too many: {n}')"
    
    parsed_39 = grammar_39.parse(walrus_code)
    assert parsed_39.get_code() == walrus_code
    assert list(grammar_39.iter_errors(parsed_39)) == [], list(grammar_39.iter_errors(parsed_39))
    print("✓ Walrus operator parsed with Python 3.9 grammar")
    errors_36 = [(e.code, e.start_pos) for e in grammar_36.iter_errors(grammar_36.parse(walrus_code))]
    assert errors_36 == [(901, (1, 6))], errors_36
    print(f"✓ Python 3.6 grammar rejects the walrus operator as expected: {errors_36}")
    
    # Test node properties
    simple_code = "x = 42"
    parsed_simple = parso.parse(simple_code)
    leaf = parsed_simple.get_first_leaf()
    print(f"\nFirst leaf: '{leaf.value}' (type: {leaf.type})")
    print(f"Leaf position: {leaf.start_pos} to {leaf.end_pos}")
    
    print("Example 5: Grammar and language features completed")
    
except Exception as e:
    print(f"Error in grammar testing: {e}")
    raise AssertionError(
        f"parso example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e